
# Face Mask Classification — Non-Mask (0) vs Mask (1)

**Assignment:** Binary image classification using a ResNet-18 trained from scratch.

### Experiments
1. **Fixed input size:** `[Batch, 3, 224, 224]`
2. **Variable input size:** `[Batch, 3, H, W]` with variable H/W

### Classes
- `0` = Non-Mask
- `1` = Mask

### Dataset
Kaggle: Face Mask 12K Images Dataset

The notebook uses **5,000 images from each class = 10,000 images total**, followed by an **80/20 train-test split**.


In [ ]:

# Install required packages if necessary
%pip install -q torch torchvision scikit-learn matplotlib seaborn pillow


In [ ]:

import os
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 1. Dataset Setup

### Option A — Kaggle/Colab

If you are using Google Colab, the easiest method is to download the dataset with the Kaggle API.

Upload your `kaggle.json` API token when prompted.


In [ ]:

# OPTIONAL: Run this cell only in Google Colab if you want to download from Kaggle.
#
# from google.colab import files
# files.upload()   # Select kaggle.json
#
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d ashishjangra27/face-mask-12k-images-dataset -p /content/mask_dataset
# !unzip -q /content/mask_dataset/face-mask-12k-images-dataset.zip -d /content/mask_dataset/extracted


In [ ]:

# Set this to the extracted dataset directory.
#
# For Colab, a common location after the previous cell is:
# DATA_ROOT = Path("/content/mask_dataset/extracted")
#
# For Windows, replace this with your actual extracted dataset path.

DATA_ROOT = Path("/content/mask_dataset/extracted")

print("Dataset root:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())


In [ ]:

# Search recursively for directories that look like mask/non-mask class folders.

if DATA_ROOT.exists():
    all_dirs = [p for p in DATA_ROOT.rglob("*") if p.is_dir()]
    for p in all_dirs[:100]:
        print(p)
else:
    print("Dataset directory does not exist yet.")
    print("Set DATA_ROOT to the folder containing the extracted Kaggle dataset.")



## 2. Locate Mask and Non-Mask Images

The Kaggle dataset may contain several folders such as `Train`, `Validation`, and `Test`.
This notebook searches recursively for image files and identifies the two classes from folder names.

If the automatic detection does not find the folders correctly, set `MASK_DIR` and `NON_MASK_DIR` manually.


In [ ]:

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files_in(folder):
    return [
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

# First, look for directories whose names contain mask / non-mask.
candidate_dirs = [p for p in DATA_ROOT.rglob("*") if p.is_dir()]

mask_candidates = [
    p for p in candidate_dirs
    if "mask" in p.name.lower() and "without" not in p.name.lower()
]

nonmask_candidates = [
    p for p in candidate_dirs
    if any(x in p.name.lower() for x in [
        "non_mask", "non-mask", "nomask", "no_mask",
        "without_mask", "without-mask", "without mask"
    ])
]

print("Mask candidates:")
for p in mask_candidates:
    print(" ", p, "->", len(image_files_in(p)), "images")

print("\nNon-mask candidates:")
for p in nonmask_candidates:
    print(" ", p, "->", len(image_files_in(p)), "images")


In [ ]:

# IMPORTANT:
# Inspect the output above.
#
# If automatic selection is incorrect, manually set these two variables.
#
# Example:
# MASK_DIR = DATA_ROOT / "Face Mask Dataset" / "Train" / "WithMask"
# NON_MASK_DIR = DATA_ROOT / "Face Mask Dataset" / "Train" / "WithoutMask"

if mask_candidates and nonmask_candidates:
    # Prefer the candidate with the largest number of images.
    MASK_DIR = max(mask_candidates, key=lambda p: len(image_files_in(p)))
    NON_MASK_DIR = max(nonmask_candidates, key=lambda p: len(image_files_in(p)))
else:
    MASK_DIR = None
    NON_MASK_DIR = None

print("Selected MASK_DIR:", MASK_DIR)
print("Selected NON_MASK_DIR:", NON_MASK_DIR)

if MASK_DIR is not None:
    print("Mask images found:", len(image_files_in(MASK_DIR)))
if NON_MASK_DIR is not None:
    print("Non-mask images found:", len(image_files_in(NON_MASK_DIR)))



## 3. Select Exactly 5,000 Images Per Class

We randomly select:

- 5,000 mask images
- 5,000 non-mask images

Total = **10,000 images**.

Then we split the complete balanced dataset into:

- 80% training = **8,000**
- 20% testing = **2,000**


In [ ]:

NUM_PER_CLASS = 5000

mask_files = image_files_in(MASK_DIR)
nonmask_files = image_files_in(NON_MASK_DIR)

if len(mask_files) < NUM_PER_CLASS:
    raise ValueError(f"Only {len(mask_files)} mask images found; need {NUM_PER_CLASS}.")

if len(nonmask_files) < NUM_PER_CLASS:
    raise ValueError(f"Only {len(nonmask_files)} non-mask images found; need {NUM_PER_CLASS}.")

rng = random.Random(SEED)

mask_files = rng.sample(mask_files, NUM_PER_CLASS)
nonmask_files = rng.sample(nonmask_files, NUM_PER_CLASS)

records = (
    [{"path": str(p), "label": 1} for p in mask_files] +
    [{"path": str(p), "label": 0} for p in nonmask_files]
)

df = pd.DataFrame(records)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Total images:", len(df))
print("\nClass counts:")
print(df["label"].value_counts().sort_index())


In [ ]:

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training images:", len(train_df))
print("Testing images :", len(test_df))

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTesting class distribution:")
print(test_df["label"].value_counts().sort_index())


## 4. Display Sample Images

In [ ]:

def show_samples(dataframe, n=12):
    sample = dataframe.sample(n=min(n, len(dataframe)), random_state=SEED)

    plt.figure(figsize=(14, 9))

    for i, (_, row) in enumerate(sample.iterrows()):
        img = Image.open(row["path"]).convert("RGB")

        ax = plt.subplot(3, 4, i + 1)
        ax.imshow(img)
        ax.set_title("Mask (1)" if row["label"] == 1 else "Non-Mask (0)")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples(df, 12)



# Experiment 1 — Fixed Input Size

Every image is transformed to:

`[3, 224, 224]`

A batch therefore has shape:

`[Batch, 3, 224, 224]`

ResNet-18 is created **without ImageNet pretrained weights**.


In [ ]:

class FixedMaskDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label


FIXED_SIZE = 224

fixed_train_transform = transforms.Compose([
    transforms.Resize((FIXED_SIZE, FIXED_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

fixed_test_transform = transforms.Compose([
    transforms.Resize((FIXED_SIZE, FIXED_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

fixed_train_dataset = FixedMaskDataset(train_df, fixed_train_transform)
fixed_test_dataset = FixedMaskDataset(test_df, fixed_test_transform)

BATCH_SIZE = 32

fixed_train_loader = DataLoader(
    fixed_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

fixed_test_loader = DataLoader(
    fixed_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

x_batch, y_batch = next(iter(fixed_train_loader))

print("Batch image shape:", x_batch.shape)
print("Batch label shape:", y_batch.shape)


In [ ]:

def build_resnet18_from_scratch(num_classes=2):
    # weights=None is important:
    # the assignment requires training from scratch.
    model = resnet18(weights=None)

    # Replace the ImageNet 1000-class classifier.
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


fixed_model = build_resnet18_from_scratch(num_classes=2).to(device)

print(fixed_model)


In [ ]:

criterion = nn.CrossEntropyLoss()

LEARNING_RATE = 1e-3
EPOCHS = 10

fixed_optimizer = torch.optim.Adam(
    fixed_model.parameters(),
    lr=LEARNING_RATE
)


In [ ]:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

    return (
        running_loss / total,
        correct / total,
        np.array(all_labels),
        np.array(all_predictions)
    )


def fit_model(model, train_loader, test_loader, criterion, optimizer,
              device, epochs=10):

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        test_loss, test_acc, _, _ = evaluate(
            model, test_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch [{epoch+1:02d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"Test Acc: {test_acc:.4f}"
        )

    return history


In [ ]:

fixed_history = fit_model(
    fixed_model,
    fixed_train_loader,
    fixed_test_loader,
    criterion,
    fixed_optimizer,
    device,
    epochs=EPOCHS
)


In [ ]:

def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["test_loss"], label="Test Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["test_acc"], label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_history(fixed_history, "Experiment 1: Fixed 224×224")


In [ ]:

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()

    labels = []
    predictions = []

    for images, y in loader:
        images = images.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        labels.extend(y.numpy())
        predictions.extend(preds.cpu().numpy())

    return np.array(labels), np.array(predictions)


fixed_y_true, fixed_y_pred = get_predictions(
    fixed_model,
    fixed_test_loader,
    device
)

print("Fixed-size accuracy:",
      accuracy_score(fixed_y_true, fixed_y_pred))

print("\nClassification Report:")
print(
    classification_report(
        fixed_y_true,
        fixed_y_pred,
        target_names=["Non-Mask (0)", "Mask (1)"],
        digits=4
    )
)

cm = confusion_matrix(fixed_y_true, fixed_y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Non-Mask (0)", "Mask (1)"],
    yticklabels=["Non-Mask (0)", "Mask (1)"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Experiment 1 — Confusion Matrix")
plt.show()



# Experiment 2 — Variable Input Size

The model must accept different spatial dimensions:

```text
[Batch, 3, H, W]
```

The key architectural component is:

```python
nn.AdaptiveAvgPool2d((1, 1))
```

This converts the final convolutional feature map to a fixed `1 × 1` spatial representation regardless of the input H/W.

### Important batching issue

PyTorch cannot stack images with different H/W into a normal batch. Therefore this experiment uses a **custom collate function that pads images within each batch to the largest H/W in that batch**.

The original images are **not resized to one fixed size**.


In [ ]:

class VariableMaskDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(row["label"])

        return image, label


variable_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

variable_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


variable_train_dataset = VariableMaskDataset(
    train_df,
    variable_train_transform
)

variable_test_dataset = VariableMaskDataset(
    test_df,
    variable_test_transform
)


In [ ]:

def variable_collate_fn(batch):
    images, labels = zip(*batch)

    # Each image is [C, H, W].
    channels = images[0].shape[0]

    max_h = max(img.shape[1] for img in images)
    max_w = max(img.shape[2] for img in images)

    padded_images = []

    for img in images:
        c, h, w = img.shape

        padded = torch.zeros(
            channels,
            max_h,
            max_w,
            dtype=img.dtype
        )

        # Center the original image inside the padded image.
        top = (max_h - h) // 2
        left = (max_w - w) // 2

        padded[:, top:top+h, left:left+w] = img

        padded_images.append(padded)

    return torch.stack(padded_images), torch.tensor(labels)


variable_train_loader = DataLoader(
    variable_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=variable_collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

variable_test_loader = DataLoader(
    variable_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=variable_collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

x_variable, y_variable = next(iter(variable_train_loader))

print("Variable experiment batch shape:", x_variable.shape)
print("Labels shape:", y_variable.shape)


In [ ]:

class VariableInputResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # ResNet-18 architecture from scratch.
        base = resnet18(weights=None)

        self.features = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4
        )

        # This is what allows arbitrary H/W.
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x


variable_model = VariableInputResNet18(num_classes=2).to(device)

print(variable_model)


In [ ]:

# Verify that the variable-size model works with different H/W.

variable_model.eval()

test_sizes = [
    (128, 128),
    (224, 224),
    (256, 320),
    (300, 400)
]

with torch.no_grad():
    for h, w in test_sizes:
        dummy = torch.randn(2, 3, h, w).to(device)
        output = variable_model(dummy)

        print(
            f"Input: {tuple(dummy.shape)} -> "
            f"Output: {tuple(output.shape)}"
        )


In [ ]:

variable_optimizer = torch.optim.Adam(
    variable_model.parameters(),
    lr=LEARNING_RATE
)

variable_history = fit_model(
    variable_model,
    variable_train_loader,
    variable_test_loader,
    criterion,
    variable_optimizer,
    device,
    epochs=EPOCHS
)


In [ ]:

plot_history(
    variable_history,
    "Experiment 2: Variable H×W"
)


In [ ]:

variable_y_true, variable_y_pred = get_predictions(
    variable_model,
    variable_test_loader,
    device
)

variable_accuracy = accuracy_score(
    variable_y_true,
    variable_y_pred
)

print("Variable-size accuracy:", variable_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        variable_y_true,
        variable_y_pred,
        target_names=["Non-Mask (0)", "Mask (1)"],
        digits=4
    )
)

cm = confusion_matrix(
    variable_y_true,
    variable_y_pred
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Non-Mask (0)", "Mask (1)"],
    yticklabels=["Non-Mask (0)", "Mask (1)"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Experiment 2 — Confusion Matrix")
plt.show()



# 5. Compare Both Experiments


In [ ]:

fixed_accuracy = accuracy_score(
    fixed_y_true,
    fixed_y_pred
)

comparison = pd.DataFrame({
    "Experiment": [
        "Fixed input (224×224)",
        "Variable input (H×W)"
    ],
    "Test Accuracy": [
        fixed_accuracy,
        variable_accuracy
    ]
})

comparison


In [ ]:

plt.figure(figsize=(8, 5))

plt.bar(
    comparison["Experiment"],
    comparison["Test Accuracy"]
)

plt.ylim(0, 1)
plt.ylabel("Test Accuracy")
plt.title("Fixed vs Variable Input Size")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()



# 6. Demonstrate the Two Input Requirements

### Experiment 1

The fixed model receives exactly:

```text
[Batch, 3, 224, 224]
```

### Experiment 2

The variable model can receive different H/W dimensions, for example:

```text
[2, 3, 128, 128]
[2, 3, 224, 224]
[2, 3, 256, 320]
[2, 3, 300, 400]
```

The `AdaptiveAvgPool2d((1,1))` layer makes the final classifier independent of H/W.

### Output

The final layer contains exactly two logits:

```text
0 → Non-Mask
1 → Mask
```

During training, `CrossEntropyLoss` is used. For probability output, softmax can be applied during inference.


In [ ]:

# Example of converting the two output logits to probabilities.

@torch.no_grad()
def predict_one(model, image_tensor):
    model.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)

    logits = model(image_tensor)

    probabilities = torch.softmax(logits, dim=1)

    predicted_class = probabilities.argmax(dim=1).item()

    return predicted_class, probabilities.squeeze(0).cpu().numpy()


# Example using one test image from the fixed-size dataset.
image_tensor, true_label = fixed_test_dataset[0]

predicted_class, probabilities = predict_one(
    fixed_model,
    image_tensor
)

class_names = ["Non-Mask (0)", "Mask (1)"]

print("True class     :", class_names[true_label])
print("Predicted class:", class_names[predicted_class])
print("Probability:")
print("  Non-Mask:", f"{probabilities[0]:.4f}")
print("  Mask    :", f"{probabilities[1]:.4f}")



# 7. Save the Trained Models

The model files can be used later for prediction on new mask/non-mask images.


In [ ]:

OUTPUT_DIR = Path("trained_models")
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(
    fixed_model.state_dict(),
    OUTPUT_DIR / "resnet18_fixed_224.pth"
)

torch.save(
    variable_model.state_dict(),
    OUTPUT_DIR / "resnet18_variable_input.pth"
)

print("Saved:")
print(OUTPUT_DIR / "resnet18_fixed_224.pth")
print(OUTPUT_DIR / "resnet18_variable_input.pth")



# 8. Assignment Conclusion

### Expected discussion

**Experiment 1 — Fixed Input**

All images are resized to `224 × 224`, so every batch has a fixed tensor shape:

`[B, 3, 224, 224]`.

This makes batching simple and usually gives efficient GPU training.

**Experiment 2 — Variable Input**

Images retain their original spatial dimensions. Since images in one batch may have different dimensions, the custom collate function pads them to a common batch size. The ResNet-18 feature extractor is followed by `AdaptiveAvgPool2d((1,1))`, allowing the network to work with different H/W values.

**Training from scratch**

Both ResNet-18 models use:

```python
resnet18(weights=None)
```

Therefore, ImageNet pretrained weights are not used.

**Output**

The final fully connected layer has two outputs:

- class `0` = Non-Mask
- class `1` = Mask

Softmax converts the two logits into class probabilities.

> Record the actual accuracy, precision, recall, F1-score, and confusion matrices produced by your run in your assignment report.
